In [1]:
import sys
import os
import torch

sys.path.append('..')

from src.model.sinpos_encoding import SinusoidalPositionalEncoding

print("Модули загружены!")

Модули загружены!


In [2]:
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Фиксируем seed
set_seed(42)

In [3]:
# cинусоидальное позиционное кодирование

d_model = 128
max_len = 512
enc = SinusoidalPositionalEncoding(d_model, max_len)
print(f"Создан модуль: Размерность эмбеддингов d_model={d_model}, Максимальная длина текста={max_len}")

# Тестовые данные
batch_size = 2
seq_len = 10
x = torch.randn(batch_size, seq_len, d_model)

# segment_ids (позиции для каждой последовательности)
# В первой строке: два объекта (по 5 токенов)
# Во второй строке: два объекта (3 и 7 токенов)
segment_ids = torch.tensor([
    [0, 1, 2, 3, 4, 0, 1, 2, 3, 4],
    [0, 1, 2, 0, 1, 2, 3, 4, 5, 6]
])

print(f" x.shape = {x.shape}")
print(f" segment_ids.shape = {segment_ids.shape}")
print(f" Первая строка: {segment_ids[0].tolist()}")
print(f" Вторая строка: {segment_ids[1].tolist()}")

output = enc(x, segment_ids)

print("=" * 50)
# Первый батч, первая позиция (токен на позиции 0)
pos_0 = output[0, 0, :5].detach().numpy()
# Первый батч, вторая позиция (токен на позиции 1)
pos_1 = output[0, 1, :5].detach().numpy()
print(f"Вектор для позиции 0 (первые 5 значений): {pos_0}")
print(f"Вектор для позиции 1 (первые 5 значений): {pos_1}")

Создан модуль: Размерность эмбеддингов d_model=128, Максимальная длина текста=512
 x.shape = torch.Size([2, 10, 128])
 segment_ids.shape = torch.Size([2, 10])
 Первая строка: [0, 1, 2, 3, 4, 0, 1, 2, 3, 4]
 Вторая строка: [0, 1, 2, 0, 1, 2, 3, 4, 5, 6]
Вектор для позиции 0 (первые 5 значений): [ 1.9269153  2.4872842  0.9007172 -1.105521   0.6784184]
Вектор для позиции 1 (первые 5 значений): [ 2.7726316   1.5521662  -0.6746861  -0.48195398  0.5455268 ]


In [4]:
# Многоглавое маскированное внимание

from src.model.attention import MultiHeadMaskedAttention

d_model = 128
n_heads = 4
attn = MultiHeadMaskedAttention(d_model=d_model, n_heads=n_heads, dropout=0.1)

print(f" Модуль внимания создан:")
print(f"   - Размерность (d_model): {d_model}")
print(f"   - Количество голов (n_heads): {n_heads}")
print(f"   - Размер одной головы (d_k): {d_model // n_heads}")

# Тестовые данные
batch_size = 2
seq_len = 10
x = torch.randn(batch_size, seq_len, d_model)

# segment_ids: две последовательности в батче
segment_ids = torch.tensor([
    [1, 1, 1, 1, 1, 2, 2, 2, 2, 2],  # 5 токенов из объекта 1, 5 из объекта 2
    [1, 1, 1, 2, 2, 2, 2, 2, 2, 2]   # 3 токена из объекта 1, 7 из объекта 2
])

print(f"   Первая строка: {segment_ids[0].tolist()}")
print(f"   Вторая строка: {segment_ids[1].tolist()}")

output = attn(x, segment_ids)

diff = (output - x).abs().max().item()

print(f"   Максимальная разница с входом: {diff:.4f}")

 Модуль внимания создан:
   - Размерность (d_model): 128
   - Количество голов (n_heads): 4
   - Размер одной головы (d_k): 32
   Первая строка: [1, 1, 1, 1, 1, 2, 2, 2, 2, 2]
   Вторая строка: [1, 1, 1, 2, 2, 2, 2, 2, 2, 2]
   Максимальная разница с входом: 3.3896


In [5]:
# Feed-Forward Network 

from src.model.ffn import FeedForward

d_ff = 512
ffn = FeedForward(d_model=d_model, d_ff=d_ff, dropout=0.1)

print(f" FFN модуль создан:")
print(f"   - Размер входа (d_model): {d_model}")
print(f"   - Размер скрытого слоя (d_ff): {d_ff}")

# Тестовые данные
batch_size = 2
seq_len = 10
x = torch.randn(batch_size, seq_len, d_model)

print(f"\n Вход: x.shape = {x.shape}")

# Forward
output = ffn(x)

print(f" Выход: output.shape = {output.shape}")

diff = (output - x).abs().max().item()
print(f"   Максимальная разница с входом: {diff:.4f}")

 FFN модуль создан:
   - Размер входа (d_model): 128
   - Размер скрытого слоя (d_ff): 512

 Вход: x.shape = torch.Size([2, 10, 128])
 Выход: output.shape = torch.Size([2, 10, 128])
   Максимальная разница с входом: 3.5394


In [6]:
# слой трансформера

from src.model.transformer_layer import TransformerLayer

d_model = 128
n_heads = 4
d_ff = 512
layer = TransformerLayer(d_model=d_model, n_heads=n_heads, d_ff=d_ff, dropout=0.1)

print(f"Слой трансформера создан:")
print(f"   - Размерность (d_model): {d_model}")
print(f"   - Голов внимания (n_heads): {n_heads}")
print(f"   - Размер FFN (d_ff): {d_ff}")

# Тестовые данные
batch_size = 2
seq_len = 10
x = torch.randn(batch_size, seq_len, d_model)

segment_ids = torch.tensor([
    [1, 1, 1, 1, 1, 2, 2, 2, 2, 2],
    [1, 1, 1, 2, 2, 2, 2, 2, 2, 2]
])

print(f"\n Вход: x.shape = {x.shape}")

# Forward
output = layer(x, segment_ids)

print(f" Выход: output.shape = {output.shape}")

diff = (output - x).abs().max().item()
print(f"   Максимальная разница с входом: {diff:.4f}")

Слой трансформера создан:
   - Размерность (d_model): 128
   - Голов внимания (n_heads): 4
   - Размер FFN (d_ff): 512

 Вход: x.shape = torch.Size([2, 10, 128])
 Выход: output.shape = torch.Size([2, 10, 128])
   Максимальная разница с входом: 1.4337


In [7]:
# LM-head: слой, который превращает эмбеддинги в логиты (предсказания для каждого токена в словаре)
from src.model.lm_head import LMHead

vocab_size = 500 
lm_head = LMHead(d_model=d_model, vocab_size=vocab_size)

print(f" LM-head создан:")
print(f"   - Размер входа (d_model): {d_model}")
print(f"   - Размер словаря (vocab_size): {vocab_size}")

batch_size = 2
seq_len = 10
x = torch.randn(batch_size, seq_len, d_model)

print(f"\n Вход: x.shape = {x.shape}")

# Forward
logits = lm_head(x)

print(f" Выход (логиты): logits.shape = {logits.shape}")

# Проверяем, что логиты не являются вероятностями
print(f"   Сумма по словарю: {logits[0, 0, :10].sum().item():.4f}")

 LM-head создан:
   - Размер входа (d_model): 128
   - Размер словаря (vocab_size): 500

 Вход: x.shape = torch.Size([2, 10, 128])
 Выход (логиты): logits.shape = torch.Size([2, 10, 500])
   Сумма по словарю: -0.9303


In [8]:
# Cборка модели и LightningModule
import yaml

from src.model.lightning_module import GPTLightningModule

with open('../configs/model_config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

model = GPTLightningModule(config)
print(f" Модель создана")
print(f" Параметров: {sum(p.numel() for p in model.parameters()):,}")

# Тест forward
batch = {
    'input_ids': torch.randint(0, config['vocab_size'], (2, 10)),
    'segment_ids': torch.tensor([
        [1,1,1,1,1,2,2,2,2,2],
        [1,1,1,2,2,2,2,2,2,2]
    ])
}

logits = model(batch['input_ids'], batch['segment_ids'])
print(f" Forward: {logits.shape}")

loss = model.training_step(batch, 0)
print(f" Training step: loss = {loss.item():.4f}")

 Модель создана
 Параметров: 723,316
 Forward: torch.Size([2, 10, 500])
 Training step: loss = 6.3625


d:\Projects\Modern-neural-network-architectures\venv_lab\Lib\site-packages\pytorch_lightning\core\module.py:451: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`


In [9]:
# # ЗАПУСК ОБУЧЕНИЯ

# import yaml
# import pytorch_lightning as pl
# from pytorch_lightning.loggers import TensorBoardLogger
# from pytorch_lightning.callbacks import ModelCheckpoint

# from src.model.lightning_module import GPTLightningModule
# from src.data.data_module import GPTDataModule

# with open('../configs/model_config.yaml', 'r', encoding='utf-8') as f:
#     config = yaml.safe_load(f)

# # Данные
# dm = GPTDataModule(config)
# dm.setup()

# # Модель
# model = GPTLightningModule(config)

# # Логгер и сохранение
# logger = TensorBoardLogger('logs/', name='gpt_model')
# checkpoint = ModelCheckpoint(
#     dirpath='checkpoints/',
#     monitor='val_perplexity',
#     mode='min',
#     save_top_k=1
# )

# # Trainer и обучение
# trainer = pl.Trainer(
#     max_epochs=config.get('max_epochs', 10),
#     logger=logger,
#     callbacks=[checkpoint],
# )

# trainer.fit(model, dm)

# print(f" Готово! Лучшая val_perplexity: {checkpoint.best_model_score:.4f}")

In [14]:
import sys
sys.path.append('..')
import yaml
from src.model.lightning_module import GPTLightningModule

with open('../configs/model_config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

best_model = GPTLightningModule.load_from_checkpoint(
    'checkpoints/epoch=2-step=1590.ckpt',  # ← правильное имя!
    config=config
)
best_model.eval()
print("Модель загружена!")

Модель загружена!


In [19]:
# Генерация текста

import json
import sys
sys.path.append('..')

from src.tokenization import encode, decode

with open('../data/tokenization/bpe_vocab.json', 'r', encoding='utf-8') as f:
    vocab = json.load(f)

prompt = input("Введите текст: ")
prompt_tokens = encode(prompt, vocab)

print(f"Запрос: {prompt}")
print(f"Токены: {prompt_tokens}")

# Генерация
input_ids = torch.tensor([prompt_tokens], dtype=torch.long)
segment_ids = torch.ones_like(input_ids)

with torch.no_grad():
    for _ in range(30):
        logits = best_model(input_ids, segment_ids)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_token], dim=1)
        segment_ids = torch.ones_like(input_ids)


generated_tokens = input_ids[0].tolist()
generated_text = decode(generated_tokens, vocab)

print(f"Сгенерированный текст:\n{generated_text}")

Запрос: hello!
Токены: [254, 213, 308, 14]
Сгенерированный текст:
hello!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
